# 📚 PDF Chapter Splitter
Intelligently splits PDFs into chapters using lazy-splitter.

**Runtime → Run all** to start, then upload your PDF when prompted.

In [ ]:
# @title 1. Install & Setup
!pip install lazy-splitter -q

import os
import subprocess
import urllib.request
import urllib.parse
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display, clear_output
import js

print('✅ Dependencies ready!')

# Get session ID from URL
session_id = None
GITHUB_PAGES_URL = 'https://Danielzapirtan.github.io/isplit'

try:
    full_hash = js.document.location.hash
    if 'sessionId=' in full_hash:
        session_id = full_hash.split('sessionId=')[1].split('&')[0]
        print(f'📡 Linked to web app')
except:
    pass

def notify_progress(stage, detail, percent):
    """Send progress update to GitHub Pages via redirect"""
    if session_id:
        try:
            params = urllib.parse.urlencode({
                'sessionId': session_id,
                'stage': stage,
                'detail': detail,
                'percent': percent
            })
            url = f'{GITHUB_PAGES_URL}/notify?{params}'
            # Use fetch in background (won't redirect user)
            js.fetch(url, js.eval('{mode: "no-cors"}'))
        except Exception as e:
            print(f'  (progress sync: {e})')

def notify_complete(drive_url):
    """Notify GitHub Pages that processing is complete"""
    if session_id:
        try:
            params = urllib.parse.urlencode({
                'sessionId': session_id,
                'driveUrl': drive_url
            })
            url = f'{GITHUB_PAGES_URL}/notify?{params}'
            js.fetch(url, js.eval('{mode: "no-cors"}'))
            print('✅ Original tab updated!')
        except Exception as e:
            print(f'⚠️  Could not update web app: {e}')
            print('   Files are safe in your Google Drive!')

In [ ]:
# @title 2. Mount Google Drive & Upload
drive.mount('/content/drive')
print('✅ Google Drive mounted!')

uploader = widgets.FileUpload(
    accept='.pdf',
    multiple=False,
    description='Choose PDF'
)
display(widgets.Label('📤 Upload your PDF:'))
display(uploader)

In [ ]:
# @title 3. Preview & Split (with Progress)
if uploader.value:
    filename = list(uploader.value.keys())[0]
    file_content = uploader.value[filename]['content']
    
    input_path = f'/content/{filename}'
    with open(input_path, 'wb') as f:
        f.write(file_content)
    
    # Stage 1: Preview
    notify_progress('preview', 'Analyzing PDF structure...', 30)
    print(f'🔍 Analyzing: {filename}\n')
    
    result = subprocess.run(
        ['pdf-splitter', 'preview', input_path,
         '--strategy', 'heuristic',
         '--sensitivity', 'high'],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
    else:
        print('⚠️  Preview issue:', result.stderr[:200])
        print('Will proceed with splitting anyway...')
    
    # Stage 2: Split
    notify_progress('splitting', 'Splitting into chapters...', 50)
    
    folder_name = f'Chapters_{filename[:-4]}'
    output_dir = f'/content/drive/MyDrive/{folder_name}'
    os.makedirs(output_dir, exist_ok=True)
    
    print(f'\n✂️ Splitting: {filename}')
    print(f'📁 Output: Google Drive → {folder_name}/\n')
    
    result = subprocess.run(
        ['pdf-splitter', 'split', input_path,
         '-o', output_dir,
         '--strategy', 'heuristic',
         '--sensitivity', 'high'],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        notify_progress('saving', 'Saving to Google Drive...', 80)
        print('✅ Success!')
        print(result.stdout)
        
        chapter_files = sorted([f for f in os.listdir(output_dir) if f.endswith('.pdf')])
        print(f'\n📊 Created {len(chapter_files)} chapters:\n')
        for f in chapter_files[:20]:
            size_kb = os.path.getsize(f'{output_dir}/{f}') / 1024
            print(f'   📄 {f} ({size_kb:.0f} KB)')
        if len(chapter_files) > 20:
            print(f'   ... and {len(chapter_files)-20} more')
    else:
        print('❌ Error:', result.stderr)
        print('\n🔄 Trying with default settings...')
        
        result2 = subprocess.run(
            ['pdf-splitter', 'split', input_path, '-o', output_dir],
            capture_output=True,
            text=True
        )
        
        if result2.returncode == 0:
            notify_progress('saving', 'Saving with default settings...', 80)
            print('✅ Success with default settings!')
            print(result2.stdout)
        else:
            print('❌ Failed:', result2.stderr[:300])
else:
    print('⏳ Upload a PDF first, then re-run this cell (Runtime → Run cell)')


In [ ]:
# @title 4. Done! Notify Web App
if uploader.value:
    filename = list(uploader.value.keys())[0]
    folder_name = f'Chapters_{filename[:-4]}'
    
    drive_url = f'https://drive.google.com/drive/folders/1-{folder_name.replace(" ", "_")}'
    
    print('🎉 All done!')
    print(f'📁 Files saved to: Google Drive → {folder_name}/')
    
    if session_id:
        notify_complete(drive_url)
        print('You can close this tab now.')
    else:
        print('\n💡 Files ready in your Google Drive. Open drive.google.com to access.')
else:
    print('⚠️  Upload a PDF first')
